In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

spark = SparkSession.builder.appName('practice').getOrCreate()

my_schema = StructType([
    StructField('Age', IntegerType()),
    StructField('Gender', StringType()),
    StructField('Avg_Daily_Screen_Time_hr', DecimalType(4,2)),
    StructField('Primary_Device', StringType()),
    StructField('Exceeded_Recommended_Limit', BooleanType()),
    StructField('Educational_to_Recreational_Ratio', DecimalType(4,2)),
    StructField('Health_Impacts', StringType()),
    StructField('Urban_or_Rural', StringType())
])

df = spark.read.schema(my_schema).csv("/Volumes/mycatalog/myschema/myvolume/Indian_Kids_Screen_Time.csv", header = True)

**What is the average daily screen time by age group (e.g., 5–7, 8–10, etc.)?**

In [0]:
grouped_by_age = df.withColumn('age_group', 
                               when((col('age') >= 8) & (col('age') <= 10), '8-10')
                               .when((col('age') >= 11) & (col('age') <= 13), '11-13')
                               .when((col('age') >= 14) & (col('age') <= 16), '14-16')
                               .when((col('age') >= 17) & (col('age') <= 18), '17-18')
                               .otherwise('NA')
                                )

grouped_by_age.groupBy('age_group').agg(round(avg('avg_daily_screen_time_hr'),3).alias('avg_time')).display()

**2. Which gender and region combination (e.g., Female - Urban) shows the highest average screen time?**

In [0]:
gen_req_grp = df.groupBy('Gender', 'Urban_or_Rural').agg(avg('avg_daily_screen_time_hr').alias('avg_time'))

gen_req_grp.orderBy(col('avg_time').desc()).limit(1).display()

**3. Find the average Educational_to_Recreational_Ratio for kids who have not exceeded the recommended limit.**

In [0]:
df.where('exceeded_recommended_limit = False').agg(avg('educational_to_recreational_ratio').alias('avg')).display()

**4. What are the top 3 most common Health_Impacts reported in rural areas?**

In [0]:
df.groupBy('Health_Impacts').agg(count('*').alias('Total_Count')).orderBy(col('Total_Count').desc()).limit(3).display()

**5. Calculate the percentage of kids using each Primary_Device within each region.**

In [0]:
total_kids = df.agg(count('*').alias('Total_Kids'))

grouped = df.groupBy('urban_or_rural', 'Primary_Device').agg(count('*').alias('count')).join(total_kids)

grouped.withColumn('percentage', round(expr('count/total_kids*100'), 2)).orderBy(col('urban_or_rural'), col("Primary_Device")).display()

**6. List kids (rows) who exceed 3 hours of average screen time and have an Educational_to_Recreational_Ratio below 0.5.**

In [0]:
df.where('avg_daily_screen_time_hr > 3 and Educational_to_Recreational_Ratio < 0.5').display()

**7. Count how many kids in each age group have exceeded the recommended screen time limit.**

**8. Which primary device is most associated with health issues like "Sleep Issues"?**

In [0]:
df.where('lower(Health_Impacts) like "%sleep%"').groupBy("Primary_Device").count().orderBy(col('count').desc()).limit(1).display()

**9. What is the average screen time of kids with no reported health impacts, vs. those with any?**

In [0]:
new_group = df.withColumn('health_impacted', when(col('Health_Impacts') == 'None', 'N')
                                             .otherwise('Y'))

new_group.groupBy('health_impacted').agg(avg(col('avg_daily_screen_time_hr')).alias('avg_time')).display()

**10. Which age and gender group has the best (highest) educational-to-recreational ratio?**

In [0]:
df.groupBy('age', 'Gender').agg(avg('Educational_to_Recreational_Ratio').alias('avg_ratio')).orderBy(col('avg_ratio').desc()).limit(1).display()

**11. Rank all kids within each Urban_or_Rural group by Avg_Daily_Screen_Time_hr using dense_rank().**

In [0]:
df.withColumn('rank', dense_rank().over(Window.partitionBy('Urban_or_Rural').orderBy('Avg_Daily_Screen_Time_hr'))).display()

**12. For each Primary_Device, calculate the average screen time and assign z-scores (standardized values) to each record.**

**13. Within each Gender, find the kid with the highest screen time per device type.**

In [0]:
# df.withColumn('rank', dense_rank().over(Window.partitionBy('gender', 'Primary_Device').orderBy(col('Avg_Daily_Screen_Time_hr').desc())))

new_df = df.select('gender', 'Primary_Device', 'Avg_Daily_Screen_Time_hr', dense_rank().over(Window.partitionBy('gender', 'Primary_Device').orderBy(col('Avg_Daily_Screen_Time_hr').desc())).alias('rank'))

new_df.where('rank = 1').display()

**14. Determine for each region, how many kids are above the regional average screen time.**

In [0]:
regional_avg = df.groupBy('Urban_or_Rural').agg(avg('Avg_Daily_Screen_Time_hr').alias('avg'))

# regional_avg.display()

df.alias('a').join(regional_avg.alias('b'), col('a.Urban_or_Rural') == col('b.Urban_or_Rural')).where('a.Avg_Daily_Screen_Time_hr > avg').groupBy('a.Urban_or_Rural').count().display()

**15. For each age group, assign quartile ranks (Q1, Q2, Q3, Q4) based on screen time.**

**16. Find all records where Avg_Daily_Screen_Time_hr is greater than the national average.**

In [0]:
nat_avg = df.select(round(avg('Avg_Daily_Screen_Time_hr'), 2).alias('avg'))

df.join(nat_avg).select('age', 'Gender', 'Avg_Daily_Screen_Time_hr', 'avg').where('Avg_Daily_Screen_Time_hr > avg').display()

**17. Create a category: "Balanced", "Recreational-Heavy", or "Educational-Heavy" based on the ratio, and count how many fall into each.**

In [0]:
category = df.withColumn('category', when((col('Educational_to_Recreational_Ratio') > 0) & (col('Educational_to_Recreational_Ratio') <= 0.3),'Balanced')
                                     .when((col('Educational_to_Recreational_Ratio') > 0.3) & (col('Educational_to_Recreational_Ratio') <= 0.5),'Recreational-Heavy')
                                     .otherwise('Educational-Heavy'))

category.groupBy('category').count().display()

**18. Calculate how many kids use mobile phones as their primary device and also reported more than one health issue.**

In [0]:
df.where('Primary_Device = "Smartphone" and Health_Impacts like "%,%"').agg(count('*').alias('count')).display()

**19. List the top 5 age groups with the worst average Health_Impacts (measured by frequency).**

**20. Classify kids into screen time buckets: Low (<1.5 hrs), Moderate (1.5–3 hrs), High (3–5 hrs), and Excessive (>5 hrs), and count how many fall in each group per region.**

In [0]:
df.withColumn('screen_time', when(col('Avg_Daily_Screen_Time_hr') < 1.5, 'Low')
                             .when((col('Avg_Daily_Screen_Time_hr') >= 1.5) & (col('Avg_Daily_Screen_Time_hr') < 3), 'Moderate')
                             .when((col('Avg_Daily_Screen_Time_hr') >= 3) & (col('Avg_Daily_Screen_Time_hr') < 5), 'High')
                             .otherwise('Excessive')).groupBy('screen_time').count().display()

